# INGESTION DOC WORD

## Docx2txtLoader

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader("../data/word_files/Document_RAG_Word.docx")
docs = loader.load()


print("Nombre de documents :", len(docs))
print(docs[0].page_content[:500])
print(docs[0].metadata)

## version avancée : Quand Docx2txt ne suffit pas 

In [ ]:
from langchain_community.document_loaders import UnstructuredWordDocumentLoader

loader = UnstructuredWordDocumentLoader("../data/word_files/Document_RAG_Word.docx")
docs = loader.load()

print("Nombre de documents :", len(docs))
print(docs[0].page_content[:500])
print(docs[0].metadata)

# Atelier pratique : Nettoyage intelligent

In [ ]:
import re

def clean_word_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)      # espaces multiples
    text = re.sub(r"\n+", "\n", text)     # sauts de ligne multiples
    text = text.replace("•", "-")         # normalisation puces
    text = text.strip()
    return text


# Pipeline complet word + clean + Split 

## 3 étapes :

1. Charger le document word 
2. le nettoyer proprement
3. le découper en chunks intelligents, adaptés aux embeddings 


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def process_word_file(path):
        # 1) Charger le document Word
    docs = Docx2txtLoader(path).load()
        # 2) Nettoyer le texte extrait
    cleaned = []
    for d in docs:
        d.page_content = clean_word_text(d.page_content)
        cleaned.append(d)
        # 3) Découpage intelligent en chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = splitter.split_documents(cleaned)
    return chunks


### Application de la fonction !

In [ ]:
chunks = process_word_file("../data/word_files/Document_RAG_Word.docx")

print("Nombre de chunks :", len(chunks))
print(chunks[0].page_content)
print(chunks[0].metadata)

In [ ]:
for i, c in enumerate(chunks[:3]):
    print("="*80)
    print(f"CHUNK {i+1}")
    print("="*80)
    print(c.page_content[:500], "\n")

## Création d'une classe pour la pipeline Word

In [ ]:
class WordIngestion:
    def __init__(self, chunk_size = 1000, chunk_overlap = 150):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap,
            separators=["\n\n", "\n", ".", " ", ""]
        )

    def _clean(self, text : str) -> str:
        return clean_word_text(text)

    def process(self, path : str):
        docs = Docx2txtLoader(path).load()
        for d in docs:
            d.page_content = self._clean(d.page_content)
        return self.splitter.split_documents(docs)

# Application

In [ ]:
ingestion_word = WordIngestion(
    chunk_size=900,
    chunk_overlap=120
)

In [ ]:
chunks = ingestion_word.process("../data/word_files/Document_RAG_Word.docx")

In [ ]:
len(chunks)

In [ ]:
chunks[0].page_content


In [ ]:
chunks[0].metadata

In [ ]:
for i, c in enumerate(chunks[:3]):
    print("="*80)
    print(f"CHUNK {i+1}")
    print("="*80)
    print(c.page_content[:500], "\n")